# Lab 09 — Deploy as a Web App (Gradio)
### *Turn your AI pipeline into a usable web app with guardrails and basic monitoring.*

<a href="https://colab.research.google.com/github/Tulane-CMPS-1010-AI-Systems/course-materials/blob/main/labs/09-deployment_lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Overview
You will build a small deployed experience:
- a Python handler function that calls an LLM helper
- a Gradio UI that calls the handler
- guardrails (validation + caching + friendly errors)
- lightweight monitoring (request logs + latency averages)

Then you’ll launch the app from Colab and share the link.

---

## Learning goals
- Build and run a Gradio app from a notebook.
- Add input validation, caching, and friendly error handling.
- Log requests and compute a basic latency metric.
- Practice the scientific loop: change a deployment choice → observe latency and error rate.


## 📚 Lecture Connection

**Before starting this lab, make sure you understand:**
- From **Day 9 Lecture**: What is latency, throughput, and concurrency? ("A practical definition..." section)
- From **Day 9 Lecture**: What are common deployment failure modes? ("Common deployment failure modes" section)
- From **Day 9 Lecture**: How do we handle errors gracefully? ("Error Handling" section)
- From **Day 9 Lecture**: How do we estimate costs? ("Cost Awareness" section)
- From **Day 9 Lecture**: What is caching and why does it help? ("Caching" section)

**This lab applies:**
- Building guardrails (validation, error handling)
- Understanding costs and optimization
- Implementing caching
- Adding observability (logging, metrics)

**Key connection:** In the lecture, you learned that:
- Deployment is about making "works on my laptop" into "works for users"
- Guardrails (validation, error handling) prevent failures
- Caching reduces both latency AND costs
- Cost awareness helps you choose the right model and optimize prompts

In this lab, you'll **build** a deployed app with all these features and **measure** their impact.

If you need a refresher, review Lecture Day 9, especially sections on failure modes, error handling, cost awareness, and caching.

---



In [ ]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/Tulane-CMPS-1010-AI-Systems/course-materials.git
import sys, time, random
import numpy as np
import pandas as pd

sys.path.append("./course-materials")
from course_utils import lab11_setup, lab11_generate_reply, lab11_build_demo

lab11_setup()

## 🧩 Pre-Lab Questions

> ✏️ *Edit this cell and type your answers below each question.*

Answer in **1–2 sentences each**:

1. What is one difference between a notebook demo and a deployed app?
2. Name one reason an AI app might be slow.
3. What is one thing you should never print to the screen in a deployed app?
4. **New:** Why does caching reduce both latency AND costs?

## 🧭 Scientific Process

**Question:**  
If we add **caching and input validation** to our deployed app, how do **latency, error rate, and cost** change?

**Hypothesis:**  
I expect that:
- Caching will **decrease** latency on repeated questions (because we skip the API call)
- Caching will **decrease** costs (because we don't pay for repeated API calls)
- Input validation will **decrease** error rate (because we catch bad inputs before they cause problems)

**Experiment Plan:**  
1. Build a handler function with validation, caching, and error handling
2. Test the same question multiple times with cache OFF, then ON
3. Measure latency for each run
4. Test invalid inputs (empty, too long) and observe error handling
5. Estimate costs for cached vs non-cached requests
6. Compare results and draw conclusions

**Measurement (Metrics):**  
- **Latency:** Average response time (seconds)
- **Error rate:** Percentage of requests that fail
- **Cost:** Estimated cost per request (tokens × price)
- **Cache hit rate:** Percentage of requests served from cache

**Expected Shape:**  
- Cache ON should show lower latency and lower costs for repeated questions
- Input validation should prevent errors from bad inputs
- Error handling should show friendly messages instead of crashes


### Step 1: Input Validation

First, let's add input validation. This prevents bad inputs from causing errors.

**What to check:**
- Is the input empty?
- Is the input too long? (Long inputs cost more and might cause timeouts)

We provide the structure; you fill in a few TODOs.


In [12]:
# @title TODO: Implement handler helpers

request_logs = []
_cache = {}

def validate_text(user_text: str, max_chars: int = 1200) -> str:
    '''
    Return "OK" if valid, else return an error message string that starts with "ERROR:".

    TODO:
    - Check if input is empty (after stripping whitespace)
    - Check if input is longer than max_chars
    - Return user-friendly error messages (not technical errors!)

    Hint: Look at the lecture example for input validation.
    Use text.strip() to remove whitespace, len() to check length.
    '''
    # TODO: Your code here
    # Hint: Use text.strip() to remove whitespace, len() to check length
    raise NotImplementedError("Implement validate_text - check for empty and too-long inputs")

def log_request(user_text: str, latency_s: float, cache_hit: bool, status: str):
    '''
    Append a dict into request_logs with:
      - time (HH:MM:SS format)
      - input length (len(user_text))
      - latency_s (rounded to 3 decimals)
      - cache_hit (True/False)
      - status ("ok" or "error")

    Hint: Use time.strftime("%H:%M:%S") to format time
    '''
    # TODO: Your code here
    # Hint: Create a dictionary with the fields above, then append to request_logs
    raise NotImplementedError("Implement log_request - create a dict and append to request_logs")

def handler(user_text: str, use_cache: bool = True):
    '''
    The function Gradio will call. Returns a string answer (or friendly error).

    This function:
    1. Validates input
    2. Checks cache (if enabled)
    3. Calls the API (with error handling!)
    4. Logs the request
    '''
    # Step 1: Validate input
    verdict = validate_text(user_text)
    if verdict != "OK":
        log_request(user_text, latency_s=0.0, cache_hit=False, status="error")
        return verdict  # Return the friendly error message

    # Step 2: Check cache
    if use_cache and user_text in _cache:
        ans = _cache[user_text]
        log_request(user_text, latency_s=0.0, cache_hit=True, status="ok")
        return ans

    # Step 3: Call API (with error handling!)
    t0 = time.perf_counter()
    try:
        ans = lab11_generate_reply(user_prompt=user_text)  # helper hides API details
    except Exception as e:
        # Don't show technical errors to users!
        # Return a friendly message instead
        log_request(user_text, latency_s=0.0, cache_hit=False, status="error")
        return "Sorry, the system is temporarily unavailable. Please try again in a moment."

    dt = time.perf_counter() - t0

    # Step 4: Cache the result (if enabled)
    if use_cache:
        _cache[user_text] = ans

    # Step 5: Log the request
    log_request(user_text, latency_s=dt, cache_hit=False, status="ok")
    return ans


## 🔹 Part 2 — Experiment: Cache OFF vs ON

Let's test how caching affects latency and costs. We'll run the same question multiple times with caching disabled, then enabled.


In [ ]:
# @title Experiment: Cache OFF vs ON

question = "Explain RAG in 3 bullet points."

print("🔄 Testing cache OFF (each request calls the API)...")
# cache OFF
request_logs.clear()
_cache.clear()
for i in range(3):
    _ = handler(question, use_cache=False)
df_off = pd.DataFrame(request_logs)

print("\n🔄 Testing cache ON (repeated requests use cache)...")
# cache ON
request_logs.clear()
_cache.clear()
for i in range(3):
    _ = handler(question, use_cache=True)
df_on = pd.DataFrame(request_logs)

print("\n📊 Results:")
print("=" * 50)
print("Cache OFF:")
display(df_off)
print("\nCache ON:")
display(df_on)

print("\n📈 Summary:")
print(f"  Average latency (OFF): {df_off['latency_s'].mean():.3f}s")
print(f"  Average latency (ON):  {df_on['latency_s'].mean():.3f}s")
print(f"  Latency improvement:   {((df_off['latency_s'].mean() - df_on['latency_s'].mean()) / df_off['latency_s'].mean() * 100):.1f}%")
print(f"  Cache hit rate (ON):   {(df_on['cache_hit'].sum() / len(df_on) * 100):.1f}%")
print(f"  Error rate (ON):       {(df_on['status']=='error').mean() * 100:.1f}%")

# Cost estimation (simple)
print("\n💰 Cost Analysis:")
# Rough estimate: ~50 tokens input, ~150 tokens output per request
tokens_per_request = 200  # approximate
cost_per_request = (tokens_per_request / 1_000_000) * 0.15  # gpt-4o-mini pricing
print(f"  Cost per API call:     ${cost_per_request:.6f}")
print(f"  Total cost (OFF, 3 calls): ${cost_per_request * 3:.6f}")
print(f"  Total cost (ON, 3 calls):  ${cost_per_request * 1:.6f} (only 1 API call!)")
print(f"  Cost savings:           ${cost_per_request * 2:.6f} ({((2/3)*100):.1f}% savings)")


### 💭 Quick Reflection

> ✏️ *Edit this cell and write your answers.*

- Did caching reduce latency? By how much?
- Did caching reduce costs? By how much?
- What is one risk of caching in an AI app?



In [ ]:
# @title Test error handling

print("Test 1: Empty input")
result1 = handler("", use_cache=False)
print(f"Result: {result1}\n")

print("Test 2: Input too long (2000 characters)")
long_input = "x" * 2000
result2 = handler(long_input, use_cache=False)
print(f"Result: {result2[:100]}...\n")

print("Test 3: Normal input")
result3 = handler("What is AI?", use_cache=False)
print(f"Result: {result3[:100]}...\n")

print("✅ Error handling test complete!")
print("   - Empty inputs should show friendly error messages")
print("   - Too-long inputs should show friendly error messages")
print("   - Normal inputs should work correctly")


## 🔹 Part 3 — Test Error Handling

Before launching the app, let's test that error handling works correctly.

**Test cases:**
1. Empty input (should show friendly error)
2. Input that's too long (should show friendly error)
3. Normal input (should work)


## 🔹 Part 4 — Launch Your Web App

Now let's launch your Gradio app! This creates a web interface that anyone can use.

**Try these in your app:**
- A normal question (should work)
- An empty input (should show friendly error)
- A very long input (should show friendly error)
- The same question twice (second time should be faster due to caching!)

After testing, check the logs below to see what happened.


In [ ]:
# @title 🚀 Launch Gradio app
import gradio as gr

demo = lab11_build_demo(handler_fn=handler)
demo.launch(share=True)


---

## 🔹 Part 5 – Building a Complete Gradio App for Your Project

**Goal:** Learn to build a complete Gradio app (not just a simple demo).

### Basic Gradio Structure

A complete Gradio app has:
- **Inputs**: Text boxes, file uploads, sliders, etc.
- **Outputs**: Text, images, dataframes, etc.
- **Function**: Your Python function that processes inputs
- **Layout**: How components are arranged



In [ ]:
# @title Example: RAG App with Multiple Components

import gradio as gr

def rag_app(query, chunk_size, top_k):
    """Complete RAG pipeline in one function."""
    # 1. Chunk corpus (if needed)
    # chunks = chunk_text(corpus, chunk_size=chunk_size)

    # 2. Retrieve
    # retrieved = retrieve_top_k(query, chunks, embeddings, k=top_k)

    # 3. Generate
    # context = "\n\n".join(retrieved)
    # answer = generate_answer(query, context)

    # 4. Return multiple outputs
    # return answer, context, len(retrieved)
    pass  # Replace with your actual RAG code

# Build the interface
with gr.Blocks(title="My RAG System") as demo:
    gr.Markdown("# My RAG System")

    with gr.Row():
        with gr.Column():
            query_input = gr.Textbox(label="Your Question", placeholder="Ask something...")
            chunk_slider = gr.Slider(minimum=50, maximum=500, value=150, label="Chunk Size")
            top_k_slider = gr.Slider(minimum=1, maximum=10, value=3, label="Top-K")
            submit_btn = gr.Button("Submit")

        with gr.Column():
            answer_output = gr.Textbox(label="Answer")
            context_output = gr.Textbox(label="Retrieved Context", lines=10)
            stats_output = gr.Textbox(label="Stats")

    submit_btn.click(
        fn=rag_app,
        inputs=[query_input, chunk_slider, top_k_slider],
        outputs=[answer_output, context_output, stats_output]
    )

demo.launch(share=True)




In [ ]:
# @title Adding Multiple Tabs

# For complex apps, use tabs:

with gr.Blocks() as demo:
    with gr.Tabs():
        with gr.Tab("Query"):
            # Query interface
            query_input = gr.Textbox(label="Question")
            answer_output = gr.Textbox(label="Answer")
            # ... more components

        with gr.Tab("Settings"):
            # Settings interface
            chunk_size = gr.Slider(label="Chunk Size", minimum=50, maximum=500)
            top_k = gr.Slider(label="Top-K", minimum=1, maximum=10)
            # ... more settings

        with gr.Tab("Evaluation"):
            # Evaluation interface
            test_query = gr.Textbox(label="Test Query")
            test_result = gr.Textbox(label="Result")
            # ... more evaluation components

demo.launch(share=True)




In [ ]:
# @title Adding File Upload

# For projects that need data upload:

file_input = gr.File(label="Upload Documents", file_types=[".txt", ".pdf"])

def process_file(file):
    """Process uploaded file."""
    if file is None:
        return "No file uploaded"
    # Read file content
    with open(file.name, 'r') as f:
        content = f.read()
    return f"Processed {len(content)} characters"

demo = gr.Interface(
    fn=process_file,
    inputs=file_input,
    outputs="text"
)
demo.launch(share=True)



### Project Checklist

Your project Gradio app should have:
- [ ] Clear title and instructions
- [ ] Input fields for user queries
- [ ] Output fields showing answers
- [ ] Optional: Settings panel (chunk size, top-k, etc.)
- [ ] Optional: Evaluation panel (test queries, metrics)
- [ ] Clean, organized layout

**For your project:** Use these patterns to build a complete Gradio app for PM4!

---


## 🔹 Part 5 — View Your Request Logs

After testing your app, let's look at the logs to see what happened.


In [ ]:
# @title View request logs and metrics

if len(request_logs) > 0:
    df = pd.DataFrame(request_logs)
    print("📊 Request Logs:")
    display(df)

    print("\n📈 Summary Metrics:")
    print(f"  Total requests: {len(df)}")
    print(f"  Successful: {(df['status']=='ok').sum()}")
    print(f"  Errors: {(df['status']=='error').sum()}")
    print(f"  Cache hits: {df['cache_hit'].sum()}")
    print(f"  Average latency: {df[df['status']=='ok']['latency_s'].mean():.3f}s")
    print(f"  Error rate: {(df['status']=='error').mean() * 100:.1f}%")
else:
    print("No requests logged yet. Try using your Gradio app first!")


## 🔹 Part 6 — Deployment Checklist

Imagine you are shipping this app for real users. What else would you need?

Write a checklist with 6–10 items covering:
- **Reliability:** timeouts, retries, fallbacks
- **Safety:** refusal policy, injection defense
- **Security:** secrets management, logging redaction
- **Monitoring:** what metrics would you alert on?
- **Cost:** how would you monitor and limit costs?

> ✏️ *Edit this cell and write your checklist.*


# Optional: HF Spaces export
You can deploy the same Gradio app to [Hugging Face Spaces](https://huggingface.co/spaces).

**High-level steps:**
1. Create a new Space (Gradio).
2. Add an `app.py` that builds the UI.
3. Add `requirements.txt`.
4. Set secrets in Space settings (do not commit keys).




## Post-Lab Reflection

> ✏️ *Edit this cell.*

Answer briefly (2–4 sentences each).

1. What surprised you about deploying an AI system as a web app?
2. What is one deployment risk you didn’t think about before this lab?
3. What is one monitoring metric you would alert on for your app?


---

## 🧠 AI Usage Log

> Use this section to document any generative AI assistance (e.g., ChatGPT, Claude, Copilot) you used while completing this lab or assignment.  
> Be specific — transparency and reflection matter more than the amount of AI use.


| Tool Used | Purpose | Prompt / Context | Verification & Edits |
|------------|----------|------------------|----------------------|
| (e.g., ChatGPT (GPT-5)) | (e.g., debugging, code explanation, idea generation) | (e.g., "Why does my cosine similarity return NaN?") | (e.g., ran tests on sample input, compared with lecture code) |
| (Add rows as needed) | | | |

**Summary (2–3 sentences):**  
Briefly describe what you learned or how AI helped you think through the problem.  
Example: *AI helped me notice an off-by-one error in my indexing. I double-checked by printing intermediate results and confirmed the fix.*

---



In [ ]:
# @title ✅ Checks for Lab 11
print("Running checks...")

try:
    v1 = validate_text("")
    assert isinstance(v1, str) and v1.startswith("ERROR")
    v2 = validate_text("hello")
    assert v2 == "OK"
    print("✅ validate_text ok")
except Exception as e:
    print("❌ validate_text failed:", e)

try:
    request_logs.clear()
    log_request("hi", 0.1234, False, "ok")
    assert len(request_logs) == 1
    assert "latency_s" in request_logs[0]
    print("✅ log_request ok")
except Exception as e:
    print("❌ log_request failed:", e)

try:
    out = handler("hello", use_cache=False)
    assert isinstance(out, str)
    print("✅ handler returns a string")
except Exception as e:
    print("❌ handler failed:", e)

print("Done.")
